# Module 02 Case Study: WHO Global Health Data

**Dataset:** Synthetic WHO Global Health Observatory data (180 countries, 2000–2020)  
**Goal:** Use descriptive statistics, distributions, and hypothesis testing to
understand global health inequality.

## Questions We Will Answer

1. How is life expectancy distributed globally, and is it approximately normal?
2. Is there a statistically significant difference in life expectancy between
   high-income and low-income countries?
3. Which health indicators correlate most strongly with life expectancy?
4. How did life expectancy change from 2000 to 2020 across income groups?

---

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('SciPy:', __import__('scipy').__version__)

## 1. Generate the Synthetic WHO Dataset

We simulate realistic country-level health data calibrated to actual WHO
Global Health Observatory ranges. Data spans 180 countries across two
census years (2000 and 2020).

In [ ]:
N_COUNTRIES = 180
YEARS = [2000, 2020]

# Income group distribution (World Bank classification)
income_groups = ['Low income', 'Lower-middle income', 'Upper-middle income', 'High income']
ig_weights     = [0.28, 0.30, 0.22, 0.20]

# Simulate for each country
country_ids   = [f'C{i:03d}' for i in range(1, N_COUNTRIES + 1)]
income_group  = np.random.choice(income_groups, N_COUNTRIES, p=ig_weights)

# Base life expectancy by income group (year 2000)
le_base = {
    'Low income':           np.random.normal(52, 5, N_COUNTRIES),
    'Lower-middle income':  np.random.normal(63, 4, N_COUNTRIES),
    'Upper-middle income':  np.random.normal(71, 3, N_COUNTRIES),
    'High income':          np.random.normal(78, 2, N_COUNTRIES),
}

rows = []
for year in YEARS:
    improvement = 4.0 if year == 2020 else 0.0  # avg 4-year gain 2000→2020
    for i, (cid, ig) in enumerate(zip(country_ids, income_group)):
        le = np.clip(le_base[ig][i] + improvement + np.random.normal(0, 0.5), 35, 90)
        rows.append({
            'country_id':      cid,
            'income_group':    ig,
            'year':            year,
            'life_expectancy': round(le, 1),
            # Correlated health indicators
            'physicians_per_1k': round(np.clip(
                (le - 45) / 6 + np.random.normal(0, 0.3), 0.02, 8.0), 2),
            'health_expenditure_pct_gdp': round(np.clip(
                (le - 50) / 8 + np.random.normal(0, 0.5), 1.5, 17.0), 1),
            'infant_mortality_per_1k': round(np.clip(
                120 - (le - 40) * 1.8 + np.random.normal(0, 3), 2, 120), 1),
            'obesity_rate_pct': round(np.clip(
                (le - 55) * 0.6 + np.random.normal(0, 2), 2, 38), 1),
            'sanitation_access_pct': round(np.clip(
                (le - 40) * 1.5 + np.random.normal(0, 4), 10, 100), 1),
        })

df = pd.DataFrame(rows)
df['income_group'] = pd.Categorical(
    df['income_group'], categories=income_groups, ordered=True
)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Descriptive Statistics

### 2a. Overall summary

In [ ]:
df_2020 = df[df['year'] == 2020].copy()

print('=== Life Expectancy (2020) ===')
le = df_2020['life_expectancy']
print(f'  Mean:   {le.mean():.1f} years')
print(f'  Median: {le.median():.1f} years')
print(f'  Std dev:{le.std():.1f} years')
print(f'  IQR:    {le.quantile(0.75) - le.quantile(0.25):.1f} years')
print(f'  Min:    {le.min():.1f} years')
print(f'  Max:    {le.max():.1f} years')

### 2b. Breakdown by income group

In [ ]:
group_stats = (
    df_2020.groupby('income_group', observed=True)['life_expectancy']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean': 'Mean', 'median': 'Median', 'std': 'Std Dev', 'count': 'N'})
    .round(1)
)
print(group_stats)

## 3. Question 1 — Distribution Shape

Is life expectancy approximately normally distributed, or does income group
create a multimodal pattern?

In [ ]:
fig = plt.figure(figsize=(13, 4))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Histogram
ax1 = fig.add_subplot(gs[0])
ax1.hist(df_2020['life_expectancy'], bins=20, color='steelblue', edgecolor='white')
ax1.axvline(le.mean(), color='red', linestyle='--', label=f'Mean {le.mean():.1f}')
ax1.axvline(le.median(), color='orange', linestyle='--', label=f'Median {le.median():.1f}')
ax1.set_xlabel('Life Expectancy (years)')
ax1.set_ylabel('Count')
ax1.set_title('Global Distribution (2020)')
ax1.legend(fontsize=8)

# Box plots by income group
ax2 = fig.add_subplot(gs[1])
groups = [df_2020.loc[df_2020['income_group'] == g, 'life_expectancy'].values
          for g in income_groups]
bp = ax2.boxplot(groups, labels=['Low', 'L-Mid', 'U-Mid', 'High'], patch_artist=True)
colors = ['#d73027', '#fc8d59', '#91bfdb', '#4575b4']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('Life Expectancy (years)')
ax2.set_title('By Income Group (2020)')

# Q-Q plot (overall)
ax3 = fig.add_subplot(gs[2])
stats.probplot(df_2020['life_expectancy'], dist='norm', plot=ax3)
ax3.set_title('Q-Q Plot — Normal?')

plt.suptitle('Life Expectancy Distributions — WHO 2020', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Shapiro-Wilk normality test
stat, p = stats.shapiro(df_2020['life_expectancy'])
print(f'Shapiro-Wilk: W = {stat:.4f}, p = {p:.4f}')
print('Interpretation:', 'NOT normal (p < 0.05)' if p < 0.05 else 'Approximately normal')

**Observations:**
- The global histogram is **left-skewed / bimodal** — the income group split creates
  two clusters rather than a single bell curve.
- The Q-Q plot shows clear deviation from normality in the tails.
- Within each income group, distributions are much closer to normal (see box plots).

## 4. Question 2 — Hypothesis Test: High Income vs Low Income

In [ ]:
high = df_2020.loc[df_2020['income_group'] == 'High income', 'life_expectancy']
low  = df_2020.loc[df_2020['income_group'] == 'Low income',  'life_expectancy']

print(f'High income: n={len(high)}, mean={high.mean():.1f}, std={high.std():.1f}')
print(f'Low income:  n={len(low)},  mean={low.mean():.1f}, std={low.std():.1f}')

# Two-sample t-test (Welch's, unequal variances)
t_stat, p_value = stats.ttest_ind(high, low, equal_var=False)

# Cohen's d — effect size
pooled_std = np.sqrt((high.std()**2 + low.std()**2) / 2)
cohens_d = (high.mean() - low.mean()) / pooled_std

print(f'\nWelch t-test: t = {t_stat:.2f}, p = {p_value:.2e}')
print(f"Cohen's d = {cohens_d:.2f}  (|d| > 0.8 = large effect)")
print(f'Mean gap: {high.mean() - low.mean():.1f} years')

The p-value is vanishingly small and the effect size is very large — the
difference in life expectancy between high- and low-income countries is both
statistically significant and practically enormous (~25 years on average).

## 5. Question 3 — Correlation with Life Expectancy

In [ ]:
health_cols = [
    'physicians_per_1k', 'health_expenditure_pct_gdp',
    'infant_mortality_per_1k', 'obesity_rate_pct', 'sanitation_access_pct'
]

correlations = {}
for col in health_cols:
    r, p = stats.pearsonr(df_2020['life_expectancy'], df_2020[col])
    correlations[col] = {'r': round(r, 3), 'p': p}

corr_df = pd.DataFrame(correlations).T
corr_df['|r|'] = corr_df['r'].abs()
print(corr_df.sort_values('|r|', ascending=False).to_string())

# Visual: scatter matrix for top 3 correlated indicators
top3 = ['infant_mortality_per_1k', 'sanitation_access_pct', 'physicians_per_1k']
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

colors_map = {'Low income': '#d73027', 'Lower-middle income': '#fc8d59',
              'Upper-middle income': '#91bfdb', 'High income': '#4575b4'}

for ax, col in zip(axes, top3):
    for ig in income_groups:
        subset = df_2020[df_2020['income_group'] == ig]
        ax.scatter(subset[col], subset['life_expectancy'],
                   alpha=0.6, s=20, color=colors_map[ig], label=ig)
    r, _ = stats.pearsonr(df_2020['life_expectancy'], df_2020[col])
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Life Expectancy')
    ax.set_title(f'r = {r:.3f}')

axes[0].legend(fontsize=6, loc='upper right')
plt.suptitle('Life Expectancy vs Health Indicators (2020)', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Question 4 — Change in Life Expectancy 2000 → 2020

In [ ]:
pivot = (
    df.groupby(['income_group', 'year'], observed=True)['life_expectancy']
    .mean()
    .unstack('year')
    .rename(columns={2000: 'LE_2000', 2020: 'LE_2020'})
)
pivot['gain'] = (pivot['LE_2020'] - pivot['LE_2000']).round(1)
print(pivot.round(1))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(income_groups))
width = 0.35
bars_2000 = ax.bar(x - width/2, pivot['LE_2000'], width, label='2000', color='#91bfdb')
bars_2020 = ax.bar(x + width/2, pivot['LE_2020'], width, label='2020', color='#4575b4')
ax.bar_label(bars_2000, fmt='%.1f', padding=2, fontsize=8)
ax.bar_label(bars_2020, fmt='%.1f', padding=2, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(['Low\nIncome', 'Lower-Mid\nIncome',
                     'Upper-Mid\nIncome', 'High\nIncome'], fontsize=9)
ax.set_ylabel('Mean Life Expectancy (years)')
ax.set_title('Life Expectancy by Income Group: 2000 vs 2020')
ax.legend()
ax.set_ylim(0, 95)
plt.tight_layout()
plt.show()

## 7. Summary

| Question | Finding |
|----------|---------|
| Distribution shape | Globally non-normal — bimodal due to income group split |
| High vs low income | ~25-year gap, p ≪ 0.001, Cohen's d >> 0.8 (very large effect) |
| Strongest correlators | Infant mortality (negative), sanitation access (positive) |
| 2000 → 2020 change | All groups improved; low-income countries gained ~4 years |

**Statistical methods used in this notebook:**
- Descriptive statistics (mean, median, std, IQR) with `.describe()` and `groupby().agg()`
- Distribution visualisation: histogram, box plot, Q-Q plot
- Normality testing: Shapiro-Wilk (`scipy.stats.shapiro`)
- Two-sample t-test (`scipy.stats.ttest_ind`)
- Effect size: Cohen's d
- Pearson correlation (`scipy.stats.pearsonr`)
- Grouped longitudinal comparison with pivot tables